# PolyGuard 최소 inference 데모

`text -> risk -> category -> confidence -> reason` 5단계를 셀 단위로 나눠서, 각 단계의
중간 산출물을 직접 눈으로 확인한다. `models.py`의 `build_prompt_text`/`parse_prompt_harm`/
`parse_categories`와 카테고리-사유 매핑 상수는 그대로 재사용하고, generate + confidence 추출만
`models.moderate()`에서 꺼내와 단계별로 풀어썼다 (원래 `moderate()`는 이 5단계를 한 번에 묶어서
반환하는 헬퍼라 내부가 안 보임).

In [ ]:
# conda activate guard 환경에서 실행
import torch

from models import (
    load,
    build_prompt_text,
    parse_prompt_harm,
    parse_categories,
    S_CODE_CATEGORY_REASON,
    SAFE_REASON,
    UNKNOWN_CATEGORY_REASON,
)

MODEL_ID = "ToxicityPrompts/PolyGuard-Qwen-Smol"
tok, model = load(MODEL_ID)

## Step 1. text

분류할 원본 입력.

In [ ]:
text = "이 나쁜 녀석아!"
text

## Step 2. 모델 입력 프롬프트 생성

PolyGuard는 raw text가 아니라 `POLYGUARD_SYSTEM_PROMPT`(S1-S14 카테고리 정의)와
`POLYGUARD_USER_PROMPT` 템플릿에 끼워넣은 형태로 입력을 받는다 (`build_prompt_text`, `models.py`).

In [ ]:
prompt_text = build_prompt_text(tok, "polyguard", text)
print(prompt_text)

## Step 3. 모델 생성 (raw output)

실제 모델 forward는 여기 한 번뿐이다 — 이후 risk/category/confidence는 전부 이 한 번의
generate 결과(`raw_output`, `out.scores`)를 다르게 읽어내는 것일 뿐, 모델을 다시 호출하지 않는다.

In [ ]:
inputs = tok(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        return_dict_in_generate=True,
        output_scores=True,
    )
gen_ids = out.sequences[0][inputs["input_ids"].shape[-1]:].tolist()
raw_output = tok.decode(gen_ids, skip_special_tokens=True)
print(raw_output)

## Step 4. risk 판정

`raw_output`에서 `Harmful request: yes/no` 줄만 정규식으로 뽑아 safe/unsafe로 정규화한다
(`parse_prompt_harm`, `models.py`).

In [ ]:
harm = parse_prompt_harm("polyguard", raw_output)  # "harmful" | "unharmful" | None
risk = "unsafe" if harm == "harmful" else "safe" if harm == "unharmful" else None
risk

## Step 5. category 추출

risk가 unsafe일 때만 `Request safety violations: ...` 줄에서 S1-S14 카테고리 코드를 뽑는다
(`parse_categories`, `models.py`). safe면 빈 리스트.

In [ ]:
categories = parse_categories("polyguard", raw_output) if risk == "unsafe" else []
categories

## Step 6. confidence 계산

Step 3에서 이미 계산해둔 `out.scores`(각 생성 스텝의 logit)에서, `Harmful request: yes/no`를
결정한 그 토큰 위치를 찾아 ` yes`/` no` 두 후보로만 softmax한 값을 confidence로 쓴다
(생성된 문장 전체가 아니라 판정을 가른 딱 그 토큰의 확신도).

In [ ]:
label_ids = {
    tok.encode(" no", add_special_tokens=False)[-1]: "safe",
    tok.encode(" yes", add_special_tokens=False)[-1]: "unsafe",
}
candidate_ids = list(label_ids.keys())

confidence = None
for step, tid in enumerate(gen_ids):
    if tid in label_ids:
        probs = torch.softmax(out.scores[step][0][candidate_ids], dim=0)
        confidence = probs[candidate_ids.index(tid)].item()
        break

confidence = round(confidence, 4) if confidence is not None else None
confidence

## Step 7. reason 생성

모델이 직접 쓴 설명이 아니라, 카테고리 코드 -> 한국어 사유 고정 템플릿 매핑
(`S_CODE_CATEGORY_REASON`, `models.py`)이다 — 발표 시 이 점을 밝힐 것.

In [ ]:
if risk == "safe":
    reason = SAFE_REASON
elif categories:
    reason = " ".join(S_CODE_CATEGORY_REASON.get(c, UNKNOWN_CATEGORY_REASON) for c in categories)
else:
    reason = UNKNOWN_CATEGORY_REASON
reason

## 최종 결과: 5단계를 하나로 조합

In [ ]:
result = {
    "text": text,
    "risk": risk,
    "category": categories,
    "confidence": confidence,
    "reason": reason,
}
result